In [10]:
import yfinance as yf
import plotly.graph_objects as go
from utils.trade import Trade

In [11]:
symbol = "EURUSD=X"
df = yf.download(symbol, period="max", interval="1h")

fast_ma = 21
slow_ma = 50

SLOW_MA = f"Slow_MA_({slow_ma})"
FAST_MA = f"Fast_MA_({fast_ma})"
ATR = f"{7} ATR"

df[SLOW_MA] = df["Close"].rolling(slow_ma).mean()
df[FAST_MA] = df["Close"].rolling(fast_ma).mean()

df[ATR] = (df['High'] - df['Low']).rolling(7).mean()

df.drop(columns=["Volume"], inplace=True)
df.columns = df.columns.droplevel("Ticker")
df.dropna(inplace=True)

df.tail()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Slow_MA_(50),Fast_MA_(21),7 ATR
Datetime,,,,,,,
2026-04-29 01:00:00+00:00,1.171509,1.171921,1.171097,1.171921,1.172134,1.170824,0.000510
2026-04-29 02:00:00+00:00,1.171097,1.172058,1.171097,1.171783,1.172118,1.170830,0.000549
2026-04-29 03:00:00+00:00,1.170960,1.171509,1.170960,1.171234,1.172082,1.170915,0.000569
2026-04-29 04:00:00+00:00,1.170960,1.171234,1.170823,1.170960,1.172041,1.170954,0.000569
2026-04-29 05:00:00+00:00,1.170275,1.170960,1.170275,1.170960,1.171981,1.170941,0.000627


In [ ]:
rr = 2

equity_curve = []
positions = []

for i in range(1, len(df)):
    row = df.iloc[i]
    prev = df.iloc[i - 1]

    price = row["Close"]

    if len(positions) == 0:
        if prev[FAST_MA] < prev[SLOW_MA] and row[FAST_MA] > row[SLOW_MA]:
            atr = row[ATR]

            positions.append(Trade(
                entry=price,
                sl=price - atr,
                tp=price + atr * rr,
            ).activate())

    updated_positions = []

    for pos in positions:
        # SL hit
        if row["Low"] <= pos["sl"]:
            loss = (pos["entry"] - pos["sl"]) * pos["size"]
            balance -= loss
            continue

        # TP hit → cascade
        elif row["High"] >= pos["tp"]:
            new_entry = pos["tp"]
            new_sl = new_entry * 0.99

            risk_amount = balance * risk_per_trade
            size = risk_amount / (new_entry - new_sl)
            new_tp = new_entry + (new_entry - new_sl) * rr

            updated_positions.append({
                "entry": new_entry,
                "sl": new_sl,
                "tp": new_tp,
                "size": size
            })

            # Move ALL SLs to last SL
            for p in positions:
                p["sl"] = new_sl

            continue

        updated_positions.append(pos)
        
    positions = updated_positions

    # =========================
    # Equity calculation
    # =========================
    floating = sum((price - p["entry"]) * p["size"] for p in positions)
    equity_curve.append(balance + floating)

In [13]:
# =========================
# Plot Equity Curve
# =========================
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index[1:],
    y=equity_curve,
    mode='lines',
    name='Equity Curve'
))

fig.update_layout(
    title=f"Cascading Strategy Backtest ({symbol})",
    xaxis_title="Date",
    yaxis_title="Equity",
    template="plotly_dark",
    height=800,
    width=1000
)

fig.show()